# Loop Detection in Comparison Graph

Finds cycles of any length in the pairwise comparison graph using two strategies,
each with its own wall-clock limit.

| Strategy | Best for | Complexity |
|---|---|---|
| **1. Exact DFS** | Any length, each cycle found exactly once | O(simple paths) — exact, bounded by the time limit |
| **2. Random walk** | Any length, cheap coverage | O(steps) — probabilistic, bounded by the time limit |

How it works:

- The exact search only extends to successors that sort **after** the start node, so
  every elementary cycle is enumerated exactly once, from its alphabetically-smallest
  member. Nothing is duplicated and nothing is missed. Cycles of length 2 are included.
- Both strategies check their deadline at every loop level, so one huge SCC cannot
  overrun the budget.
- `MAX_FILTER_RANGE` keeps the largest **signed** score gaps (`loser score - winner
  score`). A positive gap means the stored score disagrees with the comparison and is
  always removed first; a negative gap means the two already agree.
- Each loop is displayed from highest to lowest DB score. The closing edge, the one
  that completes the cycle, is highlighted in gold.


In [ ]:
import time
import random
from dataclasses import dataclass, field
from tqdm import tqdm as tqdm_desc

from comfyui_image_scorer.domain.graph.chain_manager import ChainManager, tarjan_scc

from comfyui_image_scorer.domain.analysis.trueskill import (
    public_score_from_rating,
    trueskill_score_from_rating,
    Rating,
)
from comfyui_image_scorer.infrastructure.persistence.comparisons_repository import (
    list_links,
    remove_comparison,
)
from comfyui_image_scorer.infrastructure.persistence.images_repository import list_nodes
from comfyui_image_scorer.infrastructure.persistence.path_handler import find_image_path
from comfyui_image_scorer.core.observability.logger import (
    ModuleLogger,
    get_logger,
    configure_package_logging,
)

from comfyui_image_scorer.infrastructure.persistence.images_repository import (
    find_node as get_image,
    update_image_rating_state,
)

# 20 = INFO. __name__ is __main__ in a notebook and configure_package_logging
# sets that logger's level, so anything higher would drop every info record.
configure_package_logging(20)
logger: ModuleLogger = get_logger(__name__)

# ── Configuration ─────────────────────────────────────────────────────────────

# -- Strategy 1: exact cycle search (DFS) --
# Wall-clock limit. Checked per SCC, per start node, and every 4096 stack pops
# inside one start node's DFS.
EXACT_TIME_LIMIT_S: float = 10

# -- Strategy 2: random walk --
# Wall-clock limit. Checked per SCC and before every walk.
RANDOM_TIME_LIMIT_S: float = 10

# Keep only the N largest signed score gaps per loop when breaking the cycle,
# sorted from largest gap to smallest. These are exactly the comparisons that
# get removed from the database.
MAX_FILTER_RANGE = 1


## 1. Load Data

In [ ]:
from custom_nodes.comfyui_image_scorer.domain.ports.repository import ImageRow

t0 = time.perf_counter()
images: list[ImageRow] = list_nodes()
comparisons = list_links()

db_score_basic: dict[str, float] = {
    img["filename"]: float(img.get("score") or 0.0) for img in images
}

db_score: dict[str, float] = {
    img["filename"]: public_score_from_rating(
        Rating(mu_skill=img["rating_mu"], sigma_uncertainty=img["rating_sigma"])
    )
    for img in images
}
db_score: dict[str, float] = {
    img["filename"]: trueskill_score_from_rating(
        Rating(mu_skill=img["rating_mu"], sigma_uncertainty=img["rating_sigma"])
    )
    for img in images
}

all_filenames = set(db_score.keys())

logger.info(
    "Loaded %d comparisons across %d images",
    len(comparisons),
    len(all_filenames),
    start_timer=t0,
)


## 2. Build Graph and Run Tarjan SCC

In [ ]:
cm = ChainManager()
cm._reset_adjacency()
cm._build_from_comparisons(comparisons)
cm._all_filenames = all_filenames

# directed edge -> comparison dict, keeping most recent per edge
edge_to_comp: dict[tuple[str, str], dict] = {}
for comp in comparisons:
    winner = comp["winner"]
    loser = comp["filename_b"] if winner == comp["filename_a"] else comp["filename_a"]
    key = (winner, loser)
    if key not in edge_to_comp or comp["id"] > edge_to_comp[key]["id"]:
        edge_to_comp[key] = comp

t0 = time.perf_counter()
_, scc_members = tarjan_scc(cm._all_filenames, cm._worse_than)

cyclic_sccs = {sid: members for sid, members in scc_members.items() if len(members) > 1}
sizes = sorted((len(m) for m in cyclic_sccs.values()), reverse=True)
logger.info(
    "Tarjan: %d SCCs, %d cyclic covering %d nodes, top sizes %s",
    len(scc_members),
    len(cyclic_sccs),
    sum(len(m) for m in cyclic_sccs.values()),
    sizes[:20],
    start_timer=t0,
)


## 3. Cycle Search

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────


def deadline_passed(start: float, limit: float) -> bool:
    return time.perf_counter() - start >= limit


def rotate_to_highest_score(cycle_nodes: list[str]) -> list[str]:
    best_idx = max(
        range(len(cycle_nodes)), key=lambda i: db_score.get(cycle_nodes[i], 0.0)
    )
    return cycle_nodes[best_idx:] + cycle_nodes[:best_idx]


def canon_key(nodes: list[str]) -> tuple[str, ...]:
    i = nodes.index(min(nodes))
    return tuple(nodes[i:] + nodes[:i])


# Signed gap: loser score minus winner score. Positive means the stored score
# disagrees with the comparison, which is the comparison worth removing.
def score_gap(comp: dict) -> float:
    loser = (
        comp["filename_b"]
        if comp["winner"] == comp["filename_a"]
        else comp["filename_a"]
    )
    return db_score[loser] - db_score[comp["winner"]]


@dataclass
class FoundLoop:
    scc_id: int
    nodes: list[str]
    edges: list[dict | None]
    origins: set[str] = field(default_factory=set)

    @property
    def length(self) -> int:
        return len(self.nodes)

    @property
    def label(self) -> str:
        strategies = "+".join(sorted(self.origins))
        return f"SCC {self.scc_id} [{strategies}] len={self.length}"


# One record per distinct cycle. Both strategies write here, so the exact and
# random groups and the merged total always carry identical information.
all_loops: dict[tuple[str, ...], FoundLoop] = {}


def collect(strategy: str, scc_id: int, cycle: list[str]) -> None:
    nodes = rotate_to_highest_score(cycle)
    key = canon_key(nodes)
    existing = all_loops.get(key)
    if existing is not None:
        existing.origins.add(strategy)
        return
    edges = [
        edge_to_comp.get((nodes[i], nodes[(i + 1) % len(nodes)]))
        for i in range(len(nodes))
    ]
    all_loops[key] = FoundLoop(scc_id, nodes, edges, {strategy})


logger.info("Helpers ready.")


In [ ]:
# ── Strategy 1: exact cycle search (DFS) ──────────────────────────────────────
#
# From each start node, enumerate simple paths back to that start. Extensions
# are limited to successors sorting after the start node, so every elementary
# cycle is enumerated exactly once, from its alphabetically-smallest member.
# Cycles of length 2 are included; self-loops are not.
#
# The deadline is checked at every loop level so one large SCC cannot overrun.

t0 = time.perf_counter()
timed_out = False
start_total = sum(len(m) for m in cyclic_sccs.values())

with tqdm_desc(desc="DFS start nodes", unit="start", total=start_total) as pbar:
    for scc_id, members in sorted(cyclic_sccs.items(), key=lambda x: -len(x[1])):
        if deadline_passed(t0, EXACT_TIME_LIMIT_S):
            timed_out = True
            break
        node_set = set(members)
        succs = {
            u: [v for v in cm._worse_than.get(u, []) if v in node_set] for u in members
        }

        for start in members:
            if deadline_passed(t0, EXACT_TIME_LIMIT_S):
                timed_out = True
                break

            stack = [(start, [start])]
            pops = 0
            while stack:
                if pops % 4096 == 0 and deadline_passed(t0, EXACT_TIME_LIMIT_S):
                    timed_out = True
                    break
                pops += 1
                node, path = stack.pop()
                for nbr in succs.get(node, []):
                    if nbr == start and len(path) >= 2:
                        collect("exact", scc_id, path)
                    elif nbr > start and nbr not in path:
                        stack.append((nbr, path + [nbr]))
            pbar.update(1)
        if timed_out:
            break

exact_found = sum(1 for x in all_loops.values() if "exact" in x.origins)
logger.info(
    "exact: %d cycles (limit %ss, timed_out=%s)",
    exact_found,
    EXACT_TIME_LIMIT_S,
    timed_out,
    start_timer=t0,
)


In [ ]:
# ── Strategy 2: random walk ───────────────────────────────────────────────────
#
# Walk a random path inside one SCC and stop at the first revisited node,
# extracting the cycle between the two visits. Every detected cycle is kept:
# there is no length filter and no attempt count. A walk cannot exceed |SCC|
# steps because visited_idx only ever grows.

t0 = time.perf_counter()
timed_out = False
rng = random.Random()

with tqdm_desc(desc="RW walks", unit="walk") as pbar:
    for scc_id, members in sorted(cyclic_sccs.items(), key=lambda x: -len(x[1])):
        if deadline_passed(t0, RANDOM_TIME_LIMIT_S):
            timed_out = True
            break
        node_set = set(members)
        succs = {
            u: [v for v in cm._worse_than.get(u, []) if v in node_set] for u in members
        }
        eligible = [u for u in members if succs.get(u)]
        if not eligible:
            continue

        while not deadline_passed(t0, RANDOM_TIME_LIMIT_S):
            start = rng.choice(eligible)
            path = [start]
            visited_idx = {start: 0}

            while True:
                neighbors = succs.get(path[-1], [])
                if not neighbors:
                    break
                nxt = rng.choice(neighbors)
                if nxt in visited_idx:
                    collect("random", scc_id, path[visited_idx[nxt] :])
                    break
                path.append(nxt)
                visited_idx[nxt] = len(path) - 1
            pbar.update(1)

random_found = sum(1 for x in all_loops.values() if "random" in x.origins)
logger.info(
    "random: %d cycles (limit %ss, timed_out=%s)",
    random_found,
    RANDOM_TIME_LIMIT_S,
    timed_out,
    start_timer=t0,
)


### 3.5 Select which comparisons break each cycle

Keeps the `MAX_FILTER_RANGE` largest signed gaps per loop — the same comparisons
that the removal step deletes.

In [ ]:
if not all_loops:
    logger.warning("No cycles found by either strategy")

filtered_comparisons: list[tuple[FoundLoop, list[dict]]] = []
no_db_row = 0

for loop in all_loops.values():
    edges = [comp for comp in loop.edges if comp is not None]
    if not edges:
        no_db_row += 1
        continue
    edges.sort(key=score_gap, reverse=True)
    filtered_comparisons.append((loop, edges[:MAX_FILTER_RANGE]))

if no_db_row:
    logger.warning("%d loops have no surviving database row for any edge", no_db_row)
logger.info("%d loops ready to break", len(filtered_comparisons))


## 4. Report Groups and Visual Comparison Viewer

Prints the three group lengths, then shows two examples: the **longest** loop and
the **shortest** loop. Ties resolve to the first one found.

Each edge is a side-by-side image pair.  
**Left (green border)** = winner.  
**Right (red border)** = loser.  
The **closing edge** (the one that creates the contradiction) has a **gold border**.

The rows are exactly the comparisons the removal step deletes, largest score gap first.

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

_SIDE_STYLES = (("WINNER", "tab:green"), ("LOSER", "tab:red"))


def show_loop_images(loop: FoundLoop, loop_edges: list[dict]) -> None:
    n_edges = len(loop_edges)
    first = loop.nodes[0]
    last = loop.nodes[-1]
    top = loop_edges[0]
    top_loser = (
        top["filename_b"] if top["winner"] == top["filename_a"] else top["filename_a"]
    )

    fig, axes = plt.subplots(n_edges, 2, figsize=(10, 4 * n_edges), squeeze=False)
    logger.info(
        f"{loop.label}\n"
        f"first={first[:10]}... ({db_score.get(first, 0.0):.3f}) ({db_score_basic.get(first, 0.0):.3f})  "
        f"last={last[:10]}... ({db_score.get(last, 0.0):.3f}) ({db_score_basic.get(last, 0.0):.3f})\n"
        f"biggest gap={score_gap(top):+.3f}  "
        f"({top_loser[:10]}... lost to {top['winner'][:10]}...)"
    )

    for row_i, comp in enumerate(loop_edges):
        # The cycle was rotated to start at the highest-scored node, so the edge
        # that closes the loop is the one whose winner is the last node.
        is_closing = comp["winner"] == last
        winner_fn = comp["winner"]
        loser_fn = (
            comp["filename_b"]
            if comp["winner"] == comp["filename_a"]
            else comp["filename_a"]
        )

        closing_tag = "  [CLOSING EDGE]" if is_closing else ""
        row_title = f"Edge {row_i + 1} of {n_edges}{closing_tag}"
        row_title += (
            f"\ngap={score_gap(comp):+.2f}  id={comp['id']}  ts={comp['timestamp']}"
        )

        for col, filename in enumerate((winner_fn, loser_fn)):
            side_label, side_color = _SIDE_STYLES[col]
            ax = axes[row_i][col]
            path = find_image_path(filename)
            if path is None:
                ax.text(
                    0.5,
                    0.5,
                    f"not found:\n{filename}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    wrap=True,
                )
            else:
                ax.imshow(mpimg.imread(path))
            title = f"{side_label} | score={db_score.get(filename, 0.0):.3f} ({db_score_basic.get(filename, 0.0):.3f})\n{filename}"
            if col == 0:
                title = f"{row_title}\n{title}"
            ax.set_title(title, fontsize=7.5, color=side_color)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor("gold" if is_closing else side_color)
                spine.set_linewidth(4 if is_closing else 3)

    # ensure text is not blocking images
    fig.tight_layout()

    fig.subplots_adjust(hspace=0.5, wspace=0.06)
    plt.show()


exact_loops = [x for x in all_loops.values() if "exact" in x.origins]
random_loops = [x for x in all_loops.values() if "random" in x.origins]
total_loops = list(all_loops.values())
logger.info(
    "groups: exact=%d random=%d total=%d (found by both=%d)",
    len(exact_loops),
    len(random_loops),
    len(total_loops),
    len(exact_loops) + len(random_loops) - len(total_loops),
)

if not filtered_comparisons:
    logger.warning("No loops survived filtering, nothing to display")
else:
    shortest = min(filtered_comparisons, key=lambda item: item[0].length)
    longest = max(filtered_comparisons, key=lambda item: item[0].length)
    for item in {id(shortest): shortest, id(longest): longest}.values():
        show_loop_images(*item)


In [ ]:
def _persist_image_state(filename: str, data: dict) -> bool:
    return update_image_rating_state(
        filename=filename,
        score=float(data["score"]),
        rating_mu=float(data["rating_mu"]),
        rating_sigma=float(data["rating_sigma"]),
        comparison_count=int(data["comparison_count"]),
        touch_timestamp=False,
    )


In [ ]:
confirmed_pairs: list[tuple[str, str]] = []

for loop, loop_edges in filtered_comparisons:
    for comp in loop_edges:
        loser = (
            comp["filename_b"]
            if comp["winner"] == comp["filename_a"]
            else comp["filename_a"]
        )
        confirmed_pairs.append((comp["winner"], loser))

logger.info("%d confirmed loop pairs to remove from database", len(confirmed_pairs))
confirmed_pairs = list(set(confirmed_pairs))
logger.info(
    "%d unique confirmed loop pairs to remove from database", len(confirmed_pairs)
)

dry_run = False  # Set to True to preview without writing to DB
if dry_run:
    logger.info("Dry run: no files were modified")
else:
    removed = 0
    for pair in tqdm_desc(
        confirmed_pairs,
        desc="Removing from DB",
        unit="pair",
        total=len(confirmed_pairs),
        delay=3,
    ):
        winner_data = get_image(pair[0])
        loser_data = get_image(pair[1])
        if winner_data is None or loser_data is None:
            logger.warning(
                "Skipping pair %s / %s, image missing from database", pair[0], pair[1]
            )
            continue
        if not remove_comparison(filename_a=pair[0], filename_b=pair[1]):
            logger.warning(
                "No comparison row to remove for pair %s / %s", pair[0], pair[1]
            )
            continue

        winner_data["comparison_count"] -= 1
        loser_data["comparison_count"] -= 1
        _persist_image_state(pair[0], winner_data)
        _persist_image_state(pair[1], loser_data)
        removed += 1

    logger.info("Removed %d loop pairs from database", removed)
